# Análise Exploratória de Dados — Churn de Clientes Telco

**Objetivo:** Entender o dataset, identificar padrões e formular hipóteses sobre o que leva ao cancelamento.

**Dataset:** [IBM Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — 7.043 clientes, 21 features.

---


## 0. Configuração

In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.preprocessing import load_raw_data, clean_data, get_data_summary
from visualization.plots import (
    plot_churn_distribution,
    plot_numerical_distributions,
    plot_categorical_churn_rate,
    plot_correlation_heatmap,
    save_figure,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)


## 1. Carregamento e Inspeção dos Dados Brutos

In [ ]:
df_bruto = load_raw_data("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Shape: {df_bruto.shape}")
df_bruto.head()


In [ ]:
df_bruto.info()


In [ ]:
# Verificar problemas em TotalCharges
print("Valores problemáticos em TotalCharges:")
print(df_bruto[df_bruto["TotalCharges"].str.strip() == ""]["TotalCharges"].count(), "strings vazias")
print()
print("Tipo de dado de TotalCharges:", df_bruto["TotalCharges"].dtype)


## 2. Limpeza dos Dados

In [ ]:
df = clean_data(df_bruto)
resumo = get_data_summary(df)
print(f"Linhas: {resumo['n_linhas']:,} | Colunas: {resumo['n_colunas']}")
print(f"Taxa de churn: {resumo['taxa_churn']:.2%}")
print()
print("Valores ausentes após limpeza:")
print(pd.Series(resumo['valores_ausentes']).sort_values(ascending=False).head(10))


## 3. Distribuição da Variável Alvo

In [ ]:
fig = plot_churn_distribution(df["Churn"])
save_figure(fig, "../reports/figures/01_distribuicao_churn.png")
plt.show()

print(f"\nRazão de classes (maioria/minoria): {(df['Churn']==0).sum() / (df['Churn']==1).sum():.2f}x")
print("Desbalanceamento moderado — usaremos scale_pos_weight no XGBoost e divisão estratificada.")


## 4. Análise das Features Numéricas

In [ ]:
colunas_num = ["tenure", "MonthlyCharges", "TotalCharges"]
fig = plot_numerical_distributions(df, colunas_num, target="Churn")
save_figure(fig, "../reports/figures/02_distribuicoes_numericas.png")
plt.show()


In [ ]:
# Comparação estatística: churners vs não-churners
print("=== Churners (Churn=1) ===")
print(df[df["Churn"]==1][colunas_num].describe().round(2))
print()
print("=== Não-Churners (Churn=0) ===")
print(df[df["Churn"]==0][colunas_num].describe().round(2))


In [ ]:
# Tenure vs Churn — insight principal
fig, ax = plt.subplots(figsize=(10, 4))
df.groupby("tenure")["Churn"].mean().rolling(3).mean().plot(ax=ax, color="#F44336", lw=2)
ax.set_xlabel("Tempo de contrato (meses)")
ax.set_ylabel("Taxa de Churn (média móvel 3 meses)")
ax.set_title("Taxa de Churn Diminui com o Tempo de Contrato", fontsize=13, fontweight="bold")
ax.axhline(df["Churn"].mean(), color="gray", linestyle="--", alpha=0.7, label="Taxa geral de churn")
ax.legend()
plt.tight_layout()
save_figure(fig, "../reports/figures/03_tenure_taxa_churn.png")
plt.show()

print("\nInsight: clientes no primeiro ano têm ~3x mais probabilidade de cancelar.")


## 5. Análise das Features Categóricas

In [ ]:
colunas_cat = [
    "Contract", "PaymentMethod", "InternetService",
    "TechSupport", "OnlineSecurity", "PaperlessBilling",
    "Partner", "Dependents", "gender",
]
fig = plot_categorical_churn_rate(df, colunas_cat, max_cols=3)
save_figure(fig, "../reports/figures/04_taxa_churn_categoricas.png")
plt.show()


In [ ]:
# Tipo de contrato é o preditor mais forte
print("Taxa de churn por tipo de contrato:")
print(df.groupby("Contract")["Churn"].agg(["mean", "count"]).round(3))
print()
print("Clientes mensais cancelam a 42% vs 11% (1 ano) e 3% (2 anos)!")


## 6. Análise de Correlação

In [ ]:
fig = plot_correlation_heatmap(df)
save_figure(fig, "../reports/figures/05_heatmap_correlacao.png")
plt.show()


## 7. Principais Descobertas e Hipóteses

| # | Descoberta | Implicação |
|---|-----------|------------|
| 1 | **Contratos mensais** têm taxa de churn de 42% | Tipo de contrato deve ser a feature principal |
| 2 | **Tenure inicial** (0-12 meses) é alto risco | Criar faixas de tenure como feature |
| 3 | **Usuários de fibra óptica** cancelam mais (41%) | Pode indicar insatisfação com o serviço |
| 4 | **Sem TechSupport / OnlineSecurity** correlaciona com churn | Pacotes de serviços são importantes |
| 5 | **Pagamento por cheque eletrônico** tem maior churn | Método de pagamento sinaliza comprometimento |
| 6 | Desbalanceamento de classes ~27% positivo | Necessário ajuste de threshold + scale_pos_weight |

Próximo: Engenharia de Features
